# 02 — Tokenizer và chuẩn bị dữ liệu
Chạy trên Kaggle CPU, Internet bật. Chạy smoke test trước; sau khi review kết quả notebook 01 và bộ lọc, bật hai cờ chạy dữ liệu thật. Output cần được lưu thành Kaggle Notebook Version/Dataset để các notebook GPU dùng lại.

In [ ]:
from pathlib import Path
import importlib.util, json, os, subprocess, sys
REPO_URL = 'https://github.com/ngochhuy/llm-reasoning.git'
REPO_REF = ''  # Đặt commit SHA 40 ký tự đã chốt trước khi tạo dữ liệu chính thức.
ON_KAGGLE = Path('/kaggle/working').exists()
REPO_DIR = Path('/kaggle/working/llm-reasoning-repo') if ON_KAGGLE else next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'src' / 'llm_reasoning').exists()), Path.cwd())
if ON_KAGGLE and len(REPO_REF) != 40:
    raise ValueError('Đặt REPO_REF thành commit SHA 40 ký tự trước khi tạo dữ liệu trên Kaggle')
if ON_KAGGLE and not (REPO_DIR / 'src' / 'llm_reasoning').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
if ON_KAGGLE:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_REF], check=True)
if (REPO_DIR / '.git').exists():
    CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
    if ON_KAGGLE and CODE_COMMIT != REPO_REF:
        raise ValueError('Commit đang chạy khác REPO_REF')
    print('Code commit:', CODE_COMMIT)
sys.path.insert(0, str(REPO_DIR / 'src'))
from llm_reasoning.config import TokenizerConfig, make_config
from llm_reasoning.tokenizer import ByteBPETokenizer
from llm_reasoning.pretrain_data import write_source
from llm_reasoning.sft_data import prepare_sft
config = make_config('target', runtime='kaggle' if ON_KAGGLE else 'local')
ROOT = Path('/kaggle/working/llm-reasoning') if ON_KAGGLE else REPO_DIR / 'artifacts'
PRETRAIN_DIR, SFT_DIR = ROOT / 'data' / 'pretrain', ROOT / 'data' / 'sft_gsm8k'
print('Python:', sys.version.split()[0], 'output:', ROOT)

## Smoke test không cần mạng
Kiểm tra train/save/load tokenizer, Unicode, special token, EOS, block và response mask trên dữ liệu giả lập. Artifact smoke ghi vào thư mục riêng.

In [ ]:
smoke_dir = ROOT / 'smoke_data'
tok = ByteBPETokenizer(TokenizerConfig(vocab_size=512))
tok.train(['Question: How many apples? Two plus three equals five. 2 + 3 = 5.'] * 50)
sample = 'Tiếng Việt 🧮: -1,234.5 = x² <think>ordinary text</think>'
assert tok.decode(tok.encode(sample)) == sample
tagged = '<think>2 + 3 = 5</think><answer>5</answer><eos>'
assert tok.decode(tok.encode(tagged, allowed_special=True)) == tagged
tok.save(smoke_dir / 'tokenizer.json')
assert ByteBPETokenizer.load(smoke_dir / 'tokenizer.json').encode(sample) == tok.encode(sample)
pre = write_source(({'text': f'Example document {i}. 2 + 3 = 5.'} for i in range(100)), source='sample', revision='synthetic', token_target=512, tokenizer=tok, output_dir=smoke_dir / 'pretrain', block_size=32, blocks_per_shard=2)
gsm_train = [{'question': f'How many apples in example {i}?', 'answer': 'We have two, then add three. 2 + 3 = <<2+3=5>>5. #### 5'} for i in range(4)]
sft = prepare_sft(gsm_train, [{'question': 'Held-out test?', 'answer': 'Add two and three. #### 5'}], tokenizer=tok, output_dir=smoke_dir / 'sft_gsm8k', gsm_revision='synthetic', validation_count=1, max_seq_len=512)
assert pre['shards']['train']['written_tokens'] == 512
assert sft['splits']['gsm_train']['accepted'] > 0
print('Smoke passed. Vocab:', tok.vocab_size, 'pretrain tokens:', pre['source']['train_tokens'], 'SFT accepted:', sft['splits']['gsm_train']['accepted'])

## Tạo dữ liệu thật
Chỉ bật sau khi xem mẫu notebook 01. `RUN_PRETRAIN` tạo tokenizer và shard FineWeb-Edu/OpenWebMath; `RUN_SFT` dùng **chính tokenizer đã lưu**. Với lần thử nhỏ, đặt ngân sách token thấp. Dùng `main` chỉ để thử; trước lần tạo artifact chính thức, đặt `REPO_REF` là commit SHA.

In [ ]:
RUN_PRETRAIN = False    
RUN_SFT = True
DRY_RUN = True
if RUN_PRETRAIN or RUN_SFT:
    if importlib.util.find_spec('datasets') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets>=3,<5'], check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src') + os.pathsep + env.get('PYTHONPATH', '')
if RUN_PRETRAIN:
    command = [sys.executable, '-m', 'llm_reasoning.pretrain_data', '--output', str(ROOT)]
    if DRY_RUN:
        command += ['--fineweb-tokens', '100000', '--math-tokens', '25000', '--tokenizer-bytes', '200000']
    print('Preparing pretrain artifacts:', command)
    subprocess.run(command, env=env, check=True)
if RUN_SFT:
    assert (PRETRAIN_DIR / 'tokenizer.json').exists(), 'Run or attach pretrain tokenizer first'
    command = [sys.executable, '-m', 'llm_reasoning.sft_data', '--pretrain-dir', str(PRETRAIN_DIR), '--output', str(SFT_DIR)]
    print('Preparing SFT artifacts:', command)
    subprocess.run(command, env=env, check=True)
print('Full preparation flags:', RUN_PRETRAIN, RUN_SFT, 'dry run:', DRY_RUN)

## Inspect manifests
Check pretrain token targets 80M/20M and tokenizer hash. SFT uses GSM8K only: reserve 500 validation questions and keep test separate. SFT prepares all GSM8K even with DRY_RUN=True. Rebuild with the final tokenizer before training.


In [ ]:
for label, path in [('pretrain', PRETRAIN_DIR / 'manifest.json'), ('sft', SFT_DIR / 'manifest.json')]:
    if path.exists():
        manifest = json.loads(path.read_text(encoding='utf-8'))
        print(label, json.dumps(manifest, ensure_ascii=False, indent=2)[:6000])
    else:
        print(label, 'artifact not yet created:', path)
print('Save and verify artifact files before GPU training. Local:', ROOT, '| Kaggle: save a Notebook Version or Dataset.')

## Record results
- Tokenizer vocabulary and preparation time.
- FineWeb-Edu/OpenWebMath token counts.
- GSM8K train/validation counts and tokens per epoch.
- Plan 1-3 epochs selected by validation; maximum 8M processed tokens.
- Save durable artifacts before GPU training.
